# Lab 03: Feature Scaling Techniques & Convergence Conditioning

[Back to Session Overview](../README.md) · [Module 02 Index](../../README.md)

---

## Overview

In `advertising.csv`, `TV` budgets reach $296.4$ while `Radio` stays within $[0.0, 49.6]$. Mismatched feature scales elongate the squared-error loss contours, forcing gradient descent to zig-zag or require tiny learning rates.

In this notebook, you will:
1. **Solve Problem 5 (`min_max_scaling`):** Scale `TV`, `Radio`, and `Newspaper` into $[0, 1]$, store `(max, min)` training bounds for inference, and compare MSE loss before vs. after scaling.
2. **Implement All Four Standard Scaling Methods** on the 4-row reference DataFrame:
   - **Maximum Absolute Scaling** (both per-column and global-max variants)
   - **Min-Max Scaling**
   - **Mean Normalization**
   - **Z-Score Standardization**
3. **Visualize Contour Conditioning:** Compare gradient descent paths on elongated vs. normalized contour surfaces.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("..") / "data" / "advertising.csv"
assert DATA_PATH.is_file(), f"Dataset not found at {DATA_PATH}"

plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---

## 1. Problem 5: Min-Max Feature Scaling on `advertising.csv`

Min-Max scaling transforms each feature column $x$ into $[0, 1]$:

$$x_{\mathrm{new}} = \frac{x_{\mathrm{old}} - x_{\min}}{x_{\max} - x_{\min}}$$

Crucially, we must save $(x_{\max}, x_{\min})$ computed on the training set so any new inference query `(tv, radio, newspaper)` is transformed identically.


In [ ]:
def get_column(data: list[list[float]], index: int) -> list[float]:
    return [row[index] for row in data]


def min_max_scaling(
    data1: list[float], data2: list[float], data3: list[float]
) -> tuple[
    tuple[list[float], list[float], list[float]], tuple[float, float, float, float, float, float]
]:
    """Scale three feature lists to [0, 1] and return (scaled_lists, stats)."""
    max_1, max_2, max_3 = max(data1), max(data2), max(data3)
    min_1, min_2, min_3 = min(data1), min(data2), min(data3)

    s1 = [(x - min_1) / (max_1 - min_1) for x in data1]
    s2 = [(x - min_2) / (max_2 - min_2) for x in data2]
    s3 = [(x - min_3) / (max_3 - min_3) for x in data3]

    return (s1, s2, s3), (max_1, max_2, max_3, min_1, min_2, min_3)


def prepare_data_scaled(
    file_name_dataset: str | Path, scale: bool = True
) -> tuple[list[list[float]], list[float], tuple[float, ...] | None]:
    """Load advertising CSV into [1, x1, x2, x3] rows, optionally applying Min-Max scaling."""
    data = np.genfromtxt(file_name_dataset, delimiter=",", skip_header=1).tolist()
    tv_data = get_column(data, 0)
    radio_data = get_column(data, 1)
    newspaper_data = get_column(data, 2)
    sales_data = get_column(data, 3)

    stats = None
    if scale:
        (tv_data, radio_data, newspaper_data), stats = min_max_scaling(
            tv_data, radio_data, newspaper_data
        )

    x_rows = [
        [1.0, x1, x2, x3] for x1, x2, x3 in zip(tv_data, radio_data, newspaper_data, strict=True)
    ]
    return x_rows, sales_data, stats


def train_sgd_vec(
    x_rows: list[list[float]], y_vals: list[float], epoch_max: int = 50, lr: float = 1e-5
) -> tuple[list[float], list[float]]:
    weights = [0.0, 0.016992259082509283, 0.0070783670518262355, -0.002307860847821344]
    losses: list[float] = []
    n = len(y_vals)
    for _ in range(epoch_max):
        for i in range(n):
            feat = x_rows[i]
            y = y_vals[i]
            y_hat = sum(f * w for f, w in zip(feat, weights, strict=True))
            losses.append((y_hat - y) ** 2)
            grads = [2.0 * xi * (y_hat - y) for xi in feat]
            weights = [w - lr * dw for w, dw in zip(weights, grads, strict=True)]
    return weights, losses


X_raw, y_raw, _ = prepare_data_scaled(DATA_PATH, scale=False)
X_scaled, y_scaled, scale_stats = prepare_data_scaled(DATA_PATH, scale=True)

w_raw, l_raw = train_sgd_vec(X_raw, y_raw, epoch_max=50, lr=1e-5)
# With features in [0, 1], we can safely use a much larger learning rate (e.g., 1e-2)
w_scl_same_lr, l_scl_same_lr = train_sgd_vec(X_scaled, y_scaled, epoch_max=50, lr=1e-5)
w_scl_fast, l_scl_fast = train_sgd_vec(X_scaled, y_scaled, epoch_max=50, lr=1e-2)

print(f"Final 100-iter mean MSE (raw, lr=1e-5)    : {np.mean(l_raw[-100:]):.3f}")
print(f"Final 100-iter mean MSE (scaled, lr=1e-2) : {np.mean(l_scl_fast[-100:]):.3f}")

# Consistent inference scaling for TV=19.2, Radio=35.9, Newspaper=51.3
assert scale_stats is not None
max1, max2, max3, min1, min2, min3 = scale_stats
query_scaled = [
    1.0,
    (19.2 - min1) / (max1 - min1),
    (35.9 - min2) / (max2 - min2),
    (51.3 - min3) / (max3 - min3),
]
pred_scaled = sum(f * w for f, w in zip(query_scaled, w_scl_fast, strict=True))
print(f"Predicted Sales (scaled model) for (19.2, 35.9, 51.3): {pred_scaled:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(l_raw[-100:], color="#dc2626")
axes[0].set_xlabel("#iteration (last 100)")
axes[0].set_ylabel("MSE loss")
axes[0].set_title("Before Feature Scaling (lr=1e-5)")

axes[1].plot(l_scl_fast[-100:], color="#16a34a")
axes[1].set_xlabel("#iteration (last 100)")
axes[1].set_ylabel("MSE loss")
axes[1].set_title("After Min-Max Scaling (lr=1e-2)")

plt.tight_layout()
plt.show()

---

## 2. Four Feature Scaling Methods on the Reference DataFrame

We reproduce the 4-row, 4-feature benchmark from the slides:

| Index | Feature 1 | Feature 2 | Feature 3 | Feature 4 |
| :---: | :---: | :---: | :---: | :---: |
| 0 | 180000 | 110 | 18.9 | 1400 |
| 1 | 360000 | 905 | 23.4 | 1800 |
| 2 | 230000 | 230 | 14.0 | 1300 |
| 3 | 60000 | 450 | 13.5 | 1500 |


In [ ]:
df = pd.DataFrame(
    [
        [180000, 110, 18.9, 1400],
        [360000, 905, 23.4, 1800],
        [230000, 230, 14.0, 1300],
        [60000, 450, 13.5, 1500],
    ],
    columns=["Feature 1", "Feature 2", "Feature 3", "Feature 4"],
)

# 1a. Per-column Maximum Absolute Scaling
df_max_abs_col = df.copy()
for col in df_max_abs_col.columns:
    df_max_abs_col[col] = df_max_abs_col[col] / df_max_abs_col[col].abs().max()

# 1b. Global Maximum Absolute Scaling (preserves cross-column scale dominance)
global_max = max(df[col].abs().max() for col in df.columns)
df_max_abs_global = df / global_max

# 2. Min-Max Scaling
df_min_max = (df - df.min()) / (df.max() - df.min())

# 3. Mean Normalization
df_mean_norm = (df - df.mean()) / (df.max() - df.min())

# 4. Z-Score Standardization (sample std, ddof=1)
df_z_score = (df - df.mean()) / df.std()

print("=== Per-Column Max-Abs Scaling ===")
print(df_max_abs_col.round(4))
print("\n=== Min-Max Scaling ===")
print(df_min_max.round(4))
print("\n=== Mean Normalization ===")
print(df_mean_norm.round(4))
print("\n=== Z-Score Standardization ===")
print(df_z_score.round(4))

assert np.isclose(df_max_abs_col.iloc[0, 0], 0.5)
assert np.isclose(df_min_max.iloc[0, 0], 0.4)
assert np.isclose(df_z_score["Feature 1"].mean(), 0.0)
assert np.isclose(df_z_score["Feature 1"].std(), 1.0)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
df.plot(kind="bar", ax=axes[0], title="Raw Features")
df_max_abs_col.plot(kind="bar", ax=axes[1], title="Max-Abs (Per Column)")
df_min_max.plot(kind="bar", ax=axes[2], title="Min-Max [0, 1]")
df_z_score.plot(kind="bar", ax=axes[3], title="Z-Score Standardization")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

---

## 3. Why Scaling Accelerates Convergence: Contour Geometry

Below we visualize the MSE loss contours on an unscaled two-feature synthetic problem ($x_1 \sim [0, 2000]$, $x_2 \sim [1, 5]$) versus its Z-score standardized counterpart. Standardization turns narrow ellipses into nearly circular level sets, allowing gradient descent to step straight toward the minimum.


In [ ]:
rng = np.random.default_rng(42)
x1_raw = rng.uniform(500, 2000, size=60)
x2_raw = rng.uniform(1, 5, size=60)
y_syn = 0.1 * x1_raw + 20.0 * x2_raw + rng.normal(0, 5, size=60)

# Center targets to focus on (w1, w2) contours
y_c = y_syn - y_syn.mean()
x1_c = x1_raw - x1_raw.mean()
x2_c = x2_raw - x2_raw.mean()

x1_z = x1_c / x1_raw.std()
x2_z = x2_c / x2_raw.std()

w1_g = np.linspace(-0.2, 0.4, 60)
w2_g = np.linspace(-20, 60, 60)
W1, W2 = np.meshgrid(w1_g, w2_g)
Z_raw = np.mean((W1[..., None] * x1_c + W2[..., None] * x2_c - y_c) ** 2, axis=-1)

wz1_g = np.linspace(-20, 100, 60)
wz2_g = np.linspace(-40, 80, 60)
WZ1, WZ2 = np.meshgrid(wz1_g, wz2_g)
Z_std = np.mean((WZ1[..., None] * x1_z + WZ2[..., None] * x2_z - y_c) ** 2, axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].contour(W1, W2, Z_raw, levels=20, cmap="magma")
axes[0].set_xlabel("w1 (large-scale feature)")
axes[0].set_ylabel("w2 (small-scale feature)")
axes[0].set_title("Unscaled Contours (Ill-Conditioned Ellipse)")

axes[1].contour(WZ1, WZ2, Z_std, levels=20, cmap="viridis")
axes[1].set_xlabel("w1 (standardized)")
axes[1].set_ylabel("w2 (standardized)")
axes[1].set_title("Standardized Contours (Well-Conditioned)")

plt.tight_layout()
plt.show()